Four Typing Approaches in Python (since Python 3.8)
1. Duck Typing (default Python style)
“If it looks like a duck and quacks like a duck…”
Type is determined by behavior (methods available).
No explicit type declarations needed.
Very flexible, but fewer safety checks.
2. Goose Typing (using ABCs)
Based on Abstract Base Classes (ABCs).
Uses runtime checks (e.g., isinstance()).
Objects must explicitly:
Inherit from an ABC, or
Be registered with it.
More structured and strict than duck typing.
3. Static Typing
Familiar from languages like C/Java.
Introduced in Python via:
typing module (Python 3.5+)
Type hints (PEP 484)
Checked by external tools (e.g., MyPy, IDEs).
Focuses on declared types, not just behavior.
4. Static Duck Typing (Protocols)
Combines duck typing with static typing.
Based on typing.Protocol (Python 3.8+).
Type checkers verify:
Whether an object has required methods.
Doesn’t require inheritance → structural typing.
Inspired by languages like Go.

"Python Digs Sequences" (Extreme Duck Typing)
The core philosophy discussed here is that Python actively wants your code to work. It goes out of its way to treat your objects like sequences or iterables, even if you only provide the bare minimum.

If you remember the Vowels class from the previous excerpt, it only had one special method: __getitem__. It was missing __iter__ (which normally allows looping) and __contains__ (which powers the in operator).

So, how did looping and the in operator still work?
Because Python has built-in fallback mechanisms in its C-level interpreter:

Iteration Fallback: If Python needs to iterate over an object but can't find an __iter__ method, it says, "Wait, does this thing have __getitem__?" If yes, it just starts feeding it integer indexes (0, 1, 2...) until it hits an IndexError. Boom—instant iteration.

Membership Fallback (in): If you use the in operator and there is no __contains__ method, Python relies on that same iteration trick. It just does a manual, sequential scan through the object to see if the item exists.

In [1]:
class Dog:
    def speak(self):
        return "Woof!"

class Robot:
    def speak(self):
        return "Beep boop!"

# This function expects a "Dynamic Protocol" (anything that has a speak method).
# It doesn't check the type. It just tries to run the code.
def make_noise(entity):
    print(entity.speak())

dog = Dog()
robot = Robot()

# Both work perfectly! Python figures it out at runtime.
make_noise(dog)    # Prints: Woof!
make_noise(robot)  # Prints: Beep boop!

Woof!
Beep boop!


Static Protocols (The new "Type Checking" way)
Sometimes, in large apps, you want tools (like mypy) to warn you if you mess up before you even run the code. You do this by officially defining the protocol.

In [2]:
from typing import Protocol

# We explicitly define the rule: "A Speaker MUST have a speak() method"
class Speaker(Protocol):
    def speak(self) -> str:
        pass

class Cat:
    def speak(self):
        return "Meow!"

class Rock:
    pass # A rock can't speak!

# Now we tell Python: "Only accept things that follow the Speaker protocol"
def make_noise_strictly(entity: Speaker):
    print(entity.speak())

cat = Cat()
rock = Rock()

make_noise_strictly(cat)   # Code runs fine.
make_noise_strictly(rock)  # A Static Type Checker will flag an ERROR here before the code even runs!

Meow!


AttributeError: 'Rock' object has no attribute 'speak'

In [3]:
class ToyBox:
    def __init__(self):
        self.toys = ["Action Figure", "Yo-Yo", "Puzzle"]

    # We ONLY teach the class how to fetch an item by its index number
    def __getitem__(self, index):
        return self.toys[index]

my_box = ToyBox()

# 1. Fetching by index works (because we explicitly wrote __getitem__)
print(my_box[1])  # Prints: Yo-Yo


Yo-Yo


In [4]:
# 2. Looping works! 
# Python's fallback: "I don't see __iter__. I'll just keep calling __getitem__ with 0, 1, 2 until it errors out."
for toy in my_box:
    print(toy) 
# Prints: Action Figure, Yo-Yo, Puzzle


# 3. The 'in' operator works!
# Python's fallback: "I don't see __contains__. I'll just loop through the __getitem__ indexes to see if it's there."
print("Yo-Yo" in my_box)  # Prints: True

Action Figure
Yo-Yo
Puzzle
True


Monkey patching is a funny name for a very powerful trick: it means changing how a class or module works while the program is currently running, without ever touching the original source code file.

In [5]:
# The original class (imagine this is locked in a file you can't edit)
class SmartSpeaker:
    def say_hello(self):
        return "Hello there!"

# You create your speaker
alexa = SmartSpeaker()
print(alexa.say_hello())  # Prints: Hello there!

# You want it to tell a joke, but it doesn't know how!
# print(alexa.tell_joke())  <-- This would crash with an Error.

# --- THE MONKEY PATCH ---

# 1. Write the new behavior as a normal standalone function
def funny_joke(self):
    return "Why did the programmer quit his job? Because he didn't get arrays!"

# 2. Slap the new function onto the class while the program is running!
SmartSpeaker.tell_joke = funny_joke

# 3. Now the exact same object suddenly has a new ability!
print(alexa.tell_joke())  # Prints: Why did the programmer quit...

Hello there!
Why did the programmer quit his job? Because he didn't get arrays!


In [6]:
class Calculator:
    def add(self, a, b):
        # Uh oh, the original developer made a typo!
        return a - b 

math_bot = Calculator()
print(math_bot.add(10, 5))  # Prints: 5 (Wrong!)

# --- THE MONKEY PATCH ---

# 1. Write the correct function
def correct_add(self, a, b):
    return a + b

# 2. Overwrite the broken method in the original class
Calculator.add = correct_add

# Now it works perfectly!
print(math_bot.add(10, 5))  # Prints: 15

5
15


EAFP stands for "Easier to Ask Forgiveness than Permission." It is a core Python coding philosophy. It means you should just go ahead and try to execute an operation, assuming it will work. If it turns out the operation is invalid (like a missing file, an incorrect type, or a missing dictionary key), you just catch the error and handle it ("ask forgiveness").

In [ ]:
# This is LYBL
user_profile = {"name": "Alice", "city": "Cairo"}

# We Look Before We Leap: "Excuse me, do you have an 'age' key?"
if "age" in user_profile:
    age = user_profile["age"]
else:
    age = "Unknown"
    
print(age) # Prints: Unknown

Unknown


In [ ]:
# This is EAFP
user_profile = {"name": "Alice", "city": "Cairo"}

# We just try it. If it fails, we ask forgiveness in the 'except' block.
try:
    age = user_profile["age"]
except KeyError:
    age = "Unknown"

print(age) # Prints: Unknown

Unknown


The Problem with Duck Typing (Accidental Similarities)
With traditional Duck Typing, we only care if an object has a specific method. But just because two objects share a method name doesn't mean they are meant to do the same thing!

2. Enter Goose Typing (Using ABCs)
Goose typing solves this by using Abstract Base Classes (ABCs). Instead of just checking if a method exists, you check if the object officially belongs to a specific "concept" or "family" (like checking its DNA).

In [9]:
from collections.abc import Sized

class Backpack:
    def __len__(self):
        return 5

my_bag = Backpack()

# GOOSE TYPING IN ACTION:
# We check if my_bag belongs to the "Sized" family (things that have a length).
if isinstance(my_bag, Sized):
    print(f"Bag holds {len(my_bag)} items.")

Bag holds 5 items.


In [10]:
from collections.abc import Sequence

class CustomList:
    def __init__(self, data):
        self.data = data
    # ... pretend we implemented sequence methods here ...

my_list = CustomList([1, 2, 3])

# Fails! CustomList doesn't officially inherit from Sequence.
print(isinstance(my_list, Sequence))  # False

# THE GOOSE TYPING FIX: 
# We explicitly register it. We declare our intent: "This IS a Sequence."
Sequence.register(CustomList)

# Now it works perfectly without having to rewrite the CustomList class!
print(isinstance(my_list, Sequence))  # True

False
True


In [11]:
from collections.abc import MutableSequence

# We sign the contract by putting MutableSequence in parentheses
class BadPlaylist(MutableSequence):
    def __init__(self):
        self.songs = []
        
    def __len__(self): 
        return len(self.songs)
        
    # UH OH! We forgot to write __getitem__, __setitem__, __delitem__, and insert!

# The code runs fine up to this point. 
print("File loaded successfully!")

# But the moment we try to CREATE the object... CRASH!
# my_playlist = BadPlaylist()  
# TypeError: Can't instantiate abstract class BadPlaylist with abstract methods...

File loaded successfully!


In [12]:
from collections.abc import MutableSequence

class GoodPlaylist(MutableSequence):
    def __init__(self):
        self.songs = []

    # 1. We fulfill the contract by writing the 5 required methods:
    def __len__(self): return len(self.songs)
    def __getitem__(self, i): return self.songs[i]
    def __setitem__(self, i, val): self.songs[i] = val
    def __delitem__(self, i): del self.songs[i]
    def insert(self, i, val): self.songs.insert(i, val)

# Now we create the object...
my_jams = GoodPlaylist()

# 2. THE REWARD! We can use methods we NEVER WROTE!
# The ABC gives us these for free because we provided the required building blocks.
my_jams.append("Bohemian Rhapsody")  # We didn't write append!
my_jams.extend(["Imagine", "Hey Jude"]) # We didn't write extend!
my_jams.reverse()                    # We didn't write reverse!

print(my_jams[0]) # Prints: Hey Jude

Hey Jude


Goose typing might lie

In [13]:
from collections.abc import Hashable

# A tuple containing a string and a list
sneaky_tuple = ("Alice", [1, 2, 3])

# 1. We ask permission (Goose Typing)
print(isinstance(sneaky_tuple, Hashable))  # Python says: True! It's safe!

# 2. We try to use it as a dictionary key based on that promise...
# my_dict = {sneaky_tuple: "Employee Data"} 
# CRASH! TypeError: unhashable type: 'list'

True


In [14]:
from collections.abc import Iterable

class ToyBox:
    def __getitem__(self, index):
        return ["Action Figure", "Yo-Yo"][index]

my_box = ToyBox()

# 1. We ask permission (Goose Typing)
print(isinstance(my_box, Iterable)) # Python says: False! You cannot loop over this!

# 2. We try to loop over it anyway...
for toy in my_box:
    print(toy) # Prints: Action Figure, Yo-Yo. 
               # It works perfectly!

False
Action Figure
Yo-Yo


In [15]:
import abc

class SmartDevice(abc.ABC):

    # We want a property, but we want to force the child class to write it.
    @property
    @abc.abstractmethod
    def battery_level(self):
        """Must return a percentage between 0 and 100."""
        pass

    # We want a class method, but the child class must write it.
    @classmethod
    @abc.abstractmethod
    def get_device_type(cls):
        """Must return the string name of the device type."""
        pass

In [16]:
class TaskQueue(abc.ABC):
    @abc.abstractmethod
    def add_task(self, task): pass

    @abc.abstractmethod
    def get_task(self): pass

    # THE FREE METHOD: 
    # It checks if the queue is empty by trying to pull a task out. 
    # If it works, it puts it back and says "Not empty!" 
    # This is a bit clunky, but it works for ANY subclass.
    def is_empty(self):
        try:
            task = self.get_task()
            self.add_task(task) 
            return False
        except LookupError:
            return True

Lazy way subclassing

In [17]:
class LazyQueue(TaskQueue):
    def __init__(self):
        self.tasks = []
        
    def add_task(self, task):
        self.tasks.append(task)
        
    def get_task(self):
        try:
            return self.tasks.pop(0)
        except IndexError:
            raise LookupError("No tasks left!")
            
    # We didn't write is_empty(), but we can still use it!

The Overachiever Subclass

In [18]:
class FastQueue(TaskQueue):
    def __init__(self, incoming_tasks):
        # DEFENSIVE PROGRAMMING TRICK:
        # We wrap the incoming tasks in list() to make our own private copy. 
        # If we didn't do this, popping tasks out of our queue would accidentally 
        # delete data from the user's original list!
        self.tasks = list(incoming_tasks)
        
    def add_task(self, task):
        self.tasks.append(task)
        
    def get_task(self):
        try:
            return self.tasks.pop(0)
        except IndexError:
            raise LookupError("No tasks left!")
            
    # THE OVERRIDE:
    # We replace the parent's clunky is_empty with a lightning-fast one-liner.
    def is_empty(self):
        return len(self.tasks) == 0

In [ ]:
import abc

class PaymentProcessor(abc.ABC):
    @abc.abstractmethod
    def pay(self, amount): pass

# We use @register. We do NOT put PaymentProcessor in the parentheses!
@PaymentProcessor.register
class StripeAPI:
    def pay(self, amount):
        print(f"Paid ${amount} via Stripe.")

stripe = StripeAPI()

# Python believes us! It considers it a subclass.
print(isinstance(stripe, PaymentProcessor))  # True


'''
Because StripeAPI didn't actually inherit from PaymentProcessor, it doesn't get any "free methods" from the parent.
The text proves this by looking at the __mro__ (Method Resolution Order). The MRO is Python's internal list of a class's family tree. If you check StripeAPI.__mro__, 
PaymentProcessor is completely missing! It is purely a virtual relationship to make type checkers and isinstance happy.
'''

True


Structural Typing (The VIP Bouncer: __subclasshook__)
This is the final, ultimate level of Duck Typing meeting Goose Typing.

Sometimes, you don't even need to use .register. Some of Python's built-in ABCs are so smart that they just automatically recognize your class if you have the right methods.

How? By using a hidden magic method called __subclasshook__.

Imagine __subclasshook__ as a VIP Bouncer at a club. Instead of checking if your name is on the guest list (Inheritance or Registration), the bouncer just looks at your shoes. If you are wearing the right shoes (methods), you get in.

In [ ]:
from collections.abc import Sized

class WordCounter:
    def __init__(self, text):
        self.words = text.split()
        
    # The Bouncer (Sized ABC) is only looking for this specific method
    def __len__(self):
        return len(self.words)

counter = WordCounter("Hello world")

# We didn't inherit from Sized.
# We didn't register with Sized.
# But it STILL returns True!
print(isinstance(counter, Sized))  # True

'''
Behind the scenes, the Sized ABC has a __subclasshook__ that says: "If the class knocking on the door has a __len__ method, automatically return True."
'''

True


### Static typing : typing ptotocol

In [ ]:
from typing import Protocol

# 1. We define the Structural Contract (The Static Duck)
class Flyable(Protocol):
    def fly(self) -> None:
        ... # (We just use '...' or 'pass' here)

class Bird:
    def fly(self): print("Flap flap")

class Dog:
    def bark(self): print("Woof")

# 2. We use the Protocol as the type hint!
def launch(entity: Flyable):
    entity.fly()

launch(Bird())  # Type Checker: PASS! (Bird has a fly method)
launch(Dog())   # Type Checker: ERROR! (Dog does not have a fly method)

In [ ]:
from typing import TypeVar, Protocol

# 1. The Protocol: "This object must support the + operator"
class Addable(Protocol):
    def __add__(self, other): ...

# 2. The TypeVar: "Let 'T' represent ANY type, AS LONG AS it follows the Addable rules."
T = TypeVar('T', bound=Addable)

# 3. The Function: "Takes type T, returns type T"
def add_together(x: T) -> T:
    return x + x

# NOW THE TYPE CHECKER IS SUPER SMART:
result_1 = add_together(5)       
# It knows result_1 is an 'int'

result_2 = add_together("Hello") 
# It knows result_2 is a 'str'
result_2

'''
ecause TypeVar remembers that "Hello" went in, it tells the editor that a string came out. When you type my_word., your editor instantly suggests .upper() and knows the code is perfectly safe.
'''

'HelloHello'

In [ ]:
# If you try to write a method that returns its own class before the class has finished loading, Python gets confused.
# I don't think it is needed in python 3.14 or on jupyter note book
from __future__ import annotations

class Robot:
    # Error! Python says "What is a Robot? I haven't finished building it yet!"
    def clone(self) -> Robot: 
        return Robot()

